In [0]:
from pyspark.sql import functions as F
import crawler.functions as cf
from trs_control_field import trs_control_field as tcf
import re
import pyspark.pandas as ps
from cni_connectors import adls_connector as adls_conn
from unicodedata import normalize
import os

caminhos

In [0]:
var_adls_uri, notebook_params = adls_conn.connect_adls()

dls = notebook_params.var_dls
adf = notebook_params.var_adf
file = notebook_params.var_file

In [0]:
lnd = dls['folders']['landing']
raw = dls['folders']['raw']

In [0]:
path = "{}".format(f'{lnd}{file["file_folder"]}').strip()
path = "{}/{}".format(var_adls_uri, path).strip()

print(f"Origem: {path}")

In [0]:
# Data de inserção
dt_insertion_raw = adf["adf_trigger_time"].split(".")[0]

# Caminho base de destino
sink = "{}".format(f'{raw}{file["raw_path"]}').strip()
var_sink_base = "{}/{}".format(var_adls_uri, sink).strip()
print(f"Destino base: {var_sink_base}")

In [0]:
Escrita

In [0]:
# Lista arquivos do diretório
files = dbutils.fs.ls(path)

In [0]:
csv_files = [f for f in files if f.name.lower().endswith(".csv")]
for file in csv_files:
  print(file.name)

In [0]:
def rename_columns(df):
  regex = re.compile(r"[',;{}()?\n\t=]")
  for col in df.columns:
    col_renamed = regex.sub('', normalize('NFKD', col.strip())
                                .encode("ASCII", "ignore")
                                .decode("ASCII")        
                                .replace("  ", "_")
                                .replace(" ", "_")
                                .replace("-", "_")
                                .replace(".", "_")
                                .replace("/", "_")
                                .replace(":", "_")
                                .replace("$","S"))
    df = df.withColumnRenamed(col, col_renamed)
  return df

In [0]:
for f in csv_files:
    file_name = f.name
    file_path = f.path

    print(f"Processando: {file_name}")

    try:
        # 1. Faz uma leitura ultrarrápida (limitada a 20 linhas) forçando UTF-8
        df_teste = spark.read \
            .option("header", "true") \
            .option("sep", ",") \
            .option("multiLine", "true") \
            .option("encoding", "UTF-8") \
            .csv(file_path) \
            .limit(20)

        # 2. Converte a amostra para string para inspecionar os caracteres
        amostra_texto = str(df_teste.collect())

        # Se houver falha de decodificação do UTF-8, o Spark gera o caractere '\ufffd' ()
        if "\ufffd" in amostra_texto:
            encoding_final = "ISO-8859-1"
        else:
            encoding_final = "UTF-8"
            
        print(f"  > Encoding definido para este arquivo: {encoding_final}")
    except Exception as e:
        # Fallback de segurança caso a amostragem falhe
        print(f"  > Erro ao testar encoding, assumindo UTF-8 por segurança. Erro: {e}")
        encoding_final = "UTF-8"

    # 1 - Ler CSV completo (Agora usando a variável encoding_final validada)
    df = spark.read \
        .option("header", "true") \
        .option("sep", ",") \
        .option("multiLine", "true") \
        .option("encoding", encoding_final) \
        .csv(file_path)
    
    # 2 - Renomear colunas
    df = rename_columns(df)

    # 3 - Extrair parte final do nome do arquivo
    match = re.search(r'ED&EED-(\w{3})(\d{2})\.csv$', file_name, re.IGNORECASE)

    if not match:
        print(f"Arquivo ignorado (padrão não encontrado): {file_name}")
        continue

    mes_abrev = match.group(1).upper()
    ano = "20" + match.group(2)

    mapa_mes = {
        "JAN": "01", "FEV": "02", "MAR": "03", "ABR": "04",
        "MAI": "05", "JUN": "06", "JUL": "07", "AGO": "08",
        "SET": "09", "OUT": "10", "NOV": "11", "DEZ": "12"
    }

    mes = mapa_mes.get(mes_abrev)

    if not mes:
        print(f"Mês inválido encontrado no arquivo: {file_name}")
        continue

    periodo = f"EED_{ano}{mes}"

    var_sink = f"{var_sink_base}{periodo}"

    print(f"Destino: {var_sink}\n")

    # 4 - Adicionar colunas de controle e gravar
    df = cf.append_control_columns(df, dt_insertion_raw)
    df.write.mode("overwrite").parquet(var_sink)
    print(f"Arquivo gravado com sucesso: {var_sink}\n")